# Notebook 08: Shor's Factoring Algorithm

---

## Overview

Shor's algorithm (1994) is arguably the most famous quantum algorithm. It factors integers in **polynomial time**, threatening the RSA cryptosystem and all public-key cryptography based on the hardness of factoring.

### What You Will Learn

1. The integer factoring problem and its connection to RSA
2. Reduction of factoring to period finding
3. Quantum period finding using the Quantum Fourier Transform (QFT)
4. Modular exponentiation circuits
5. Classical post-processing with continued fractions
6. Complete Qiskit implementation for factoring 15 and 21

### Complexity

| Algorithm | Time Complexity |
|-----------|----------------|
| Trial division | $O(\sqrt{N})$ |
| Number field sieve (best classical) | $O(\exp(c \cdot (\ln N)^{1/3}(\ln\ln N)^{2/3}))$ |
| **Shor's algorithm** | $O((\log N)^3)$ — **exponential speedup!** |

## 1. The Factoring Problem and RSA

### The Problem

Given a composite integer $N$, find non-trivial factors $p$ and $q$ such that $N = p \cdot q$.

### RSA Cryptography

RSA security relies on the assumption that factoring large numbers (typically 2048+ bits) is computationally infeasible:

1. **Key generation:** Choose two large primes $p, q$. Compute $N = pq$ (public) and $\phi(N) = (p-1)(q-1)$ (private).
2. **Encryption:** $c = m^e \mod N$ (easy with the public key $e$)
3. **Decryption:** $m = c^d \mod N$ where $d = e^{-1} \mod \phi(N)$ (needs the private key)

**Breaking RSA = Factoring $N$**, because once you know $p$ and $q$, you can compute $\phi(N)$ and hence the private key $d$.

### Shor's Threat

Shor's algorithm can factor an $n$-bit number using $O(n^3)$ quantum operations (plus $O(n^2)$ classical operations). For RSA-2048, this requires roughly 4000 logical qubits — currently beyond our hardware, but a matter of engineering, not fundamental physics.

## 2. Factoring $\to$ Period Finding (The Key Reduction)

Shor's insight: factoring can be **reduced** to the problem of finding the **period** of a function.

### The Reduction

Given $N$ to factor:

**Step 1:** Choose a random $a$ with $1 < a < N$ and $\gcd(a, N) = 1$.

(If $\gcd(a, N) > 1$, we already found a factor — lucky!)

**Step 2:** Find the **order** $r$ of $a$ modulo $N$, i.e., the smallest positive integer $r$ such that:

$$a^r \equiv 1 \pmod{N}$$

This is the **period** of the function $f(x) = a^x \mod N$.

**Step 3:** If $r$ is even, compute:

$$a^r - 1 = (a^{r/2} - 1)(a^{r/2} + 1) \equiv 0 \pmod{N}$$

**Step 4:** Compute $\gcd(a^{r/2} \pm 1, N)$. With probability $\geq 1/2$, at least one of these is a non-trivial factor of $N$.

### Example: $N = 15$, $a = 7$

| $x$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|-----|---|---|---|---|---|---|---|---|---|
| $7^x \mod 15$ | 1 | 7 | 4 | 13 | 1 | 7 | 4 | 13 | 1 |

The period is $r = 4$. Then:
- $a^{r/2} = 7^2 = 49$
- $\gcd(49 - 1, 15) = \gcd(48, 15) = 3$
- $\gcd(49 + 1, 15) = \gcd(50, 15) = 5$

So $15 = 3 \times 5$. Success!

In [ ]:
import numpy as np
from math import gcd
from fractions import Fraction
import matplotlib.pyplot as plt

# Demonstrate the periodicity of a^x mod N
def plot_modular_exponentiation(a, N, num_points=20):
    """Plot f(x) = a^x mod N to visualize the period."""
    xs = list(range(num_points))
    ys = [pow(a, x, N) for x in xs]
    
    # Find the period
    r = 1
    while pow(a, r, N) != 1:
        r += 1
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(xs, ys, color='steelblue', alpha=0.8)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel(f'${a}^x \\mod {N}$', fontsize=12)
    ax.set_title(f'Modular Exponentiation: $f(x) = {a}^x \\mod {N}$ (period $r = {r}$)', fontsize=14)
    ax.set_xticks(xs)
    
    # Highlight period
    for k in range(0, num_points, r):
        ax.axvline(x=k, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    return r

# Example 1: N=15, a=7
r = plot_modular_exponentiation(7, 15)
print(f"Period r = {r}")
print(f"gcd(7^{r//2} - 1, 15) = gcd({pow(7, r//2) - 1}, 15) = {gcd(pow(7, r//2) - 1, 15)}")
print(f"gcd(7^{r//2} + 1, 15) = gcd({pow(7, r//2) + 1}, 15) = {gcd(pow(7, r//2) + 1, 15)}")

In [ ]:
# Example 2: N=21, a=2
r = plot_modular_exponentiation(2, 21)
print(f"Period r = {r}")
if r % 2 == 0:
    print(f"gcd(2^{r//2} - 1, 21) = gcd({pow(2, r//2) - 1}, 21) = {gcd(pow(2, r//2) - 1, 21)}")
    print(f"gcd(2^{r//2} + 1, 21) = gcd({pow(2, r//2) + 1}, 21) = {gcd(pow(2, r//2) + 1, 21)}")
else:
    print(f"r is odd — need to try another base a")

## 3. Quantum Period Finding

The quantum part of Shor's algorithm solves the **order-finding problem**: given $a$ and $N$ with $\gcd(a, N) = 1$, find the smallest $r$ such that $a^r \equiv 1 \pmod{N}$.

### The Circuit

The quantum circuit uses two registers:
- **Counting register:** $t$ qubits (we need $t \geq 2\log_2 N$ for sufficient precision)
- **Work register:** $\lceil \log_2 N \rceil$ qubits

```
|0⟩^⊗t ── H^⊗t ── ┤                    ├── QFT† ── Measure
                    │  Controlled-U^(2^j) │
|1⟩       ──────── ┤                    ├──────────────────
```

### Mathematical Derivation

**Step 1:** Initialize counting register to uniform superposition:

$$|\psi_1\rangle = \frac{1}{\sqrt{2^t}} \sum_{x=0}^{2^t - 1} |x\rangle |1\rangle$$

**Step 2:** Apply controlled modular exponentiation $U_a|y\rangle = |ay \mod N\rangle$:

$$|\psi_2\rangle = \frac{1}{\sqrt{2^t}} \sum_{x=0}^{2^t - 1} |x\rangle |a^x \mod N\rangle$$

**Step 3:** Apply inverse QFT to the counting register. Since $f(x) = a^x \mod N$ is periodic with period $r$, the QFT concentrates the amplitude on values $|x\rangle$ where $x \approx k \cdot 2^t / r$ for integer $k$.

**Step 4:** Measure the counting register and use continued fractions to extract $r$.

### Why QFT Works for Period Finding

The state before QFT can be written as:

$$\frac{1}{\sqrt{2^t}} \sum_{x=0}^{2^t - 1} |x\rangle |a^x \mod N\rangle = \frac{1}{\sqrt{2^t}} \sum_{s=0}^{r-1} \left(\sum_{j} |jr + s\rangle\right) |a^s \mod N\rangle$$

The QFT acts on the periodic sum $\sum_j |jr + s\rangle$ and produces peaks at $k \cdot 2^t / r$:

$$\text{QFT}^\dagger \left(\sum_{j} |jr + s\rangle\right) \propto \sum_{k=0}^{r-1} e^{2\pi i ks/r} |k \cdot 2^t / r\rangle$$

After measurement, we obtain $m \approx k \cdot 2^t / r$, from which we can deduce $r$.

## 4. Continued Fractions — Classical Post-Processing

After measuring $m$ from the counting register, we have $m/2^t \approx k/r$ for some integer $k$. We need to extract $r$.

The **continued fraction expansion** of $m/2^t$ gives the best rational approximation $k/r$ with $r < N$.

### Algorithm

Given a real number $\alpha$:

$$\alpha = a_0 + \cfrac{1}{a_1 + \cfrac{1}{a_2 + \cfrac{1}{\ddots}}}$$

The convergents $p_k/q_k$ give the best rational approximations. We check each convergent's denominator $q_k$ as a candidate for $r$.

In [ ]:
def continued_fraction(m, Q, N):
    """
    Use continued fractions to find the period r from measurement m.
    
    Parameters:
    -----------
    m : int — measured value from counting register
    Q : int — 2^t, size of counting register
    N : int — number to factor
    
    Returns: list of candidate periods
    """
    if m == 0:
        return []  # m=0 gives no information
    
    # Use Python's Fraction for exact continued fraction
    frac = Fraction(m, Q).limit_denominator(N)
    candidates = [frac.denominator]
    
    # Also check multiples
    for k in range(2, 10):
        if frac.denominator * k < N:
            candidates.append(frac.denominator * k)
    
    return candidates

# Example: N=15, a=7, r=4
# If t=4 (Q=16), ideal measurements would be 0, 4, 8, 12
Q = 16
N = 15

print("Continued fraction analysis for N=15:")
print(f"{'m':<5} {'m/Q':<12} {'Fraction':<15} {'Candidates'}")
print("-" * 50)

for m in [0, 4, 8, 12]:
    if m == 0:
        print(f"{m:<5} {m/Q:<12.4f} {'N/A':<15} {'(no info)'}")
        continue
    frac = Fraction(m, Q).limit_denominator(N)
    candidates = continued_fraction(m, Q, N)
    print(f"{m:<5} {m/Q:<12.4f} {str(frac):<15} {candidates}")

## 5. Implementing Shor's Algorithm in Qiskit

### Strategy for Small Numbers

For the full Shor's algorithm, we need:
1. Controlled modular exponentiation: $|x\rangle|y\rangle \to |x\rangle|a^x y \mod N\rangle$
2. Inverse QFT on the counting register

For $N = 15$, we can use **compiled oracles** (pre-computed unitary matrices) since the full modular exponentiation circuit is complex.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

def qft_dagger(circuit, n, start_qubit=0):
    """
    Apply inverse QFT to qubits [start_qubit, start_qubit + n).
    """
    qubits = list(range(start_qubit, start_qubit + n))
    
    # Swap qubits (QFT convention)
    for i in range(n // 2):
        circuit.swap(qubits[i], qubits[n - 1 - i])
    
    for j in range(n):
        for m in range(j):
            circuit.cp(-np.pi / float(2**(j - m)), qubits[m], qubits[j])
        circuit.h(qubits[j])

print("Inverse QFT function defined.")

In [ ]:
def c_amod15(a, power):
    """
    Controlled multiplication by a mod 15.
    Returns a gate that performs |x⟩|y⟩ → |x⟩|a^(2^power) * y mod 15⟩ when x=1.
    
    This uses pre-compiled circuits for a = 2, 4, 7, 8, 11, 13, 14.
    """
    if a not in [2, 4, 7, 8, 11, 13, 14]:
        raise ValueError(f"a={a} not implemented")
    
    # Work register uses 4 qubits for mod 15
    U = QuantumCircuit(4)
    
    # Apply a^(2^power) mod 15
    # We compute the effective value: a^(2^power) mod 15
    effective_a = pow(a, 2**power, 15)
    
    # Multiplication by specific values mod 15 using SWAP networks
    if effective_a == 1:
        pass  # Identity
    elif effective_a == 2:
        U.swap(0, 1)
        U.swap(1, 2)
        U.swap(2, 3)
    elif effective_a == 4:
        U.swap(0, 2)
        U.swap(1, 3)
    elif effective_a == 7:
        U.swap(0, 1)
        U.swap(1, 2)
        U.swap(2, 3)
        U.x([0, 1, 2, 3])
    elif effective_a == 8:
        U.swap(2, 3)
        U.swap(1, 2)
        U.swap(0, 1)
    elif effective_a == 11:
        U.swap(2, 3)
        U.swap(1, 2)
        U.swap(0, 1)
        U.x([0, 1, 2, 3])
    elif effective_a == 13:
        U.swap(0, 2)
        U.swap(1, 3)
        U.x([0, 1, 2, 3])
    elif effective_a == 14:
        U.x([0, 1, 2, 3])
    
    U = U.to_gate()
    U.name = f"{a}^{2**power} mod 15"
    c_U = U.control()
    return c_U

print("Controlled modular exponentiation gate defined.")

In [ ]:
def shor_circuit_15(a, n_count=8):
    """
    Build Shor's circuit for factoring N=15.
    
    Parameters:
    -----------
    a : int — base for modular exponentiation
    n_count : int — number of counting qubits
    
    Returns: QuantumCircuit
    """
    # n_count counting qubits + 4 work qubits
    qc = QuantumCircuit(n_count + 4, n_count)
    
    # Initialize work register to |1⟩ (qubit n_count is LSB)
    qc.x(n_count)
    
    # Hadamard on counting register
    for q in range(n_count):
        qc.h(q)
    
    qc.barrier()
    
    # Controlled modular exponentiation
    for q in range(n_count):
        gate = c_amod15(a, q)
        qc.append(gate, [q] + list(range(n_count, n_count + 4)))
    
    qc.barrier()
    
    # Inverse QFT on counting register
    qft_dagger(qc, n_count, start_qubit=0)
    
    qc.barrier()
    
    # Measure counting register
    qc.measure(range(n_count), range(n_count))
    
    return qc

print("Shor's circuit builder defined.")

In [ ]:
# Build and visualize Shor's circuit for N=15, a=7
a = 7
N_factor = 15
n_count = 8  # counting qubits (2^8 = 256)

qc = shor_circuit_15(a, n_count)
print(f"Shor's algorithm for N={N_factor}, a={a}")
print(f"Counting qubits: {n_count} (Q = {2**n_count})")
print(f"Total qubits: {qc.num_qubits}")
print(f"Circuit depth: {qc.depth()}")

# Draw (may be large)
qc.draw('mpl', style='iqp', fold=40)
plt.title(f"Shor's Algorithm Circuit: N={N_factor}, a={a}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Run Shor's algorithm for N=15, a=7
sim = AerSimulator()
compiled = transpile(qc, sim)
result = sim.run(compiled, shots=4096).result()
counts = result.get_counts()

# Display results
print(f"\nShor's Algorithm Results (N={N_factor}, a={a})")
print("=" * 50)

plot_histogram(counts, figsize=(14, 5))
plt.title(f"Shor's Algorithm: Measurement Results (N={N_factor}, a={a})")
plt.show()

# Analyze the peaks
Q = 2**n_count
print(f"\nPeak Analysis (Q = {Q}):")
print(f"{'Measured':<12} {'Decimal':<10} {'Phase (m/Q)':<15} {'Fraction':<12} {'r candidate'}")
print("-" * 65)

# Sort by count, take top results
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

r_candidates = set()
for bitstring, count in sorted_counts[:8]:
    m = int(bitstring, 2)
    phase = m / Q
    frac = Fraction(m, Q).limit_denominator(N_factor)
    r_cand = frac.denominator
    r_candidates.add(r_cand)
    print(f"{bitstring:<12} {m:<10} {phase:<15.6f} {str(frac):<12} {r_cand}")

In [ ]:
def classical_postprocess(a, r_candidates, N):
    """
    Classical post-processing: check each candidate period r.
    """
    print(f"\nClassical Post-Processing (a={a}, N={N})")
    print("=" * 50)
    
    factors_found = set()
    
    for r in sorted(r_candidates):
        if r == 0:
            continue
        
        # Check: is a^r ≡ 1 (mod N)?
        if pow(a, r, N) != 1:
            print(f"r={r}: {a}^{r} mod {N} = {pow(a, r, N)} ≠ 1 (not the period)")
            continue
        
        print(f"r={r}: {a}^{r} mod {N} = 1 ✓")
        
        if r % 2 != 0:
            print(f"  r={r} is odd — cannot use directly")
            continue
        
        x = pow(a, r // 2, N)
        if x == N - 1:  # a^(r/2) ≡ -1 (mod N)
            print(f"  a^(r/2) mod N = {x} ≡ -1 — trivial factor")
            continue
        
        factor1 = gcd(x - 1, N)
        factor2 = gcd(x + 1, N)
        
        print(f"  a^(r/2) = {a}^{r//2} mod {N} = {x}")
        print(f"  gcd({x}-1, {N}) = gcd({x-1}, {N}) = {factor1}")
        print(f"  gcd({x}+1, {N}) = gcd({x+1}, {N}) = {factor2}")
        
        if factor1 not in [1, N]:
            factors_found.add(factor1)
        if factor2 not in [1, N]:
            factors_found.add(factor2)
    
    if factors_found:
        print(f"\n*** FACTORS FOUND: {N} = {' × '.join(map(str, sorted(factors_found)))} ***")
    else:
        print(f"\nNo non-trivial factors found. Try a different base a.")
    
    return factors_found

# Run the classical post-processing
factors = classical_postprocess(a, r_candidates, N_factor)

## 6. Factoring N = 15 with Different Bases

Let us try multiple bases $a$ and see that Shor's algorithm works for all valid choices.

In [ ]:
# Test all valid bases for N=15
N_factor = 15
valid_bases = [a for a in range(2, N_factor) if gcd(a, N_factor) == 1]

print(f"Valid bases for N={N_factor}: {valid_bases}")
print(f"(Excluded: bases with gcd(a, {N_factor}) > 1)")
print()

# For bases that our c_amod15 supports
supported_bases = [2, 4, 7, 8, 11, 13, 14]

for a in supported_bases:
    print(f"\n{'='*60}")
    print(f"Base a = {a}")
    
    # Find actual period classically (for verification)
    r_actual = 1
    while pow(a, r_actual, N_factor) != 1:
        r_actual += 1
    print(f"True period: r = {r_actual}")
    
    # Run quantum circuit
    qc = shor_circuit_15(a, n_count=8)
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=2048).result()
    counts = result.get_counts()
    
    # Extract period candidates from top measurements
    Q = 2**8
    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    
    r_candidates = set()
    for bitstring, count in sorted_counts[:6]:
        m = int(bitstring, 2)
        if m > 0:
            frac = Fraction(m, Q).limit_denominator(N_factor)
            r_candidates.add(frac.denominator)
    
    # Try each candidate
    for r in sorted(r_candidates):
        if r > 0 and pow(a, r, N_factor) == 1 and r % 2 == 0:
            x = pow(a, r // 2, N_factor)
            if x != N_factor - 1:
                f1 = gcd(x - 1, N_factor)
                f2 = gcd(x + 1, N_factor)
                if f1 not in [1, N_factor] or f2 not in [1, N_factor]:
                    print(f"  Found factors via r={r}: {N_factor} = {f1} × {f2}")
                    break
    else:
        print(f"  Could not find factors with a={a}")

## 7. The Full Shor's Algorithm — Classical Wrapper

The complete algorithm includes classical pre- and post-processing around the quantum period finding.

In [ ]:
import random

def full_shor(N, n_count=8, max_attempts=10, verbose=True):
    """
    Complete Shor's algorithm for factoring N.
    (Currently supports N=15 only due to oracle limitations.)
    """
    if verbose:
        print(f"Shor's Algorithm: Factoring N = {N}")
        print("=" * 50)
    
    # Check trivial cases
    if N % 2 == 0:
        return 2, N // 2
    
    # Check if N is a prime power
    for b in range(2, int(np.log2(N)) + 1):
        a = N**(1/b)
        if abs(a - round(a)) < 1e-10:
            return int(round(a)), N // int(round(a))
    
    sim = AerSimulator()
    
    for attempt in range(max_attempts):
        # Step 1: Choose random base
        a = random.choice([2, 4, 7, 8, 11, 13, 14])  # Valid for N=15
        
        if verbose:
            print(f"\nAttempt {attempt + 1}: a = {a}")
        
        # Check if we got lucky
        g = gcd(a, N)
        if g > 1:
            if verbose:
                print(f"  Lucky! gcd({a}, {N}) = {g}")
            return g, N // g
        
        # Step 2: Quantum period finding
        qc = shor_circuit_15(a, n_count)
        compiled = transpile(qc, sim)
        result = sim.run(compiled, shots=1024).result()
        counts = result.get_counts()
        
        # Step 3: Extract period from measurement
        Q = 2**n_count
        sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
        
        for bitstring, count in sorted_counts[:5]:
            m = int(bitstring, 2)
            if m == 0:
                continue
            
            frac = Fraction(m, Q).limit_denominator(N)
            r = frac.denominator
            
            if verbose:
                print(f"  Measured m={m}, phase={m/Q:.4f}, r_candidate={r}")
            
            # Step 4: Classical post-processing
            if r > 0 and pow(a, r, N) == 1 and r % 2 == 0:
                x = pow(a, r // 2, N)
                if x != N - 1:
                    f1 = gcd(x - 1, N)
                    f2 = gcd(x + 1, N)
                    if f1 not in [1, N] and f2 not in [1, N]:
                        if verbose:
                            print(f"  SUCCESS! r={r}, factors: {f1} × {f2}")
                        return min(f1, f2), max(f1, f2)
    
    if verbose:
        print(f"\nFailed after {max_attempts} attempts.")
    return None, None

# Factor 15
p, q = full_shor(15)
print(f"\nFinal result: 15 = {p} × {q}")

## 8. Factoring N = 21

For $N = 21 = 3 \times 7$, the modular exponentiation is more complex. We demonstrate the classical reduction and verify with a simulated quantum period finder.

In [ ]:
def classical_shor_simulation(N, verbose=True):
    """
    Simulate Shor's algorithm classically for any N.
    This demonstrates the full algorithm flow without the quantum circuit.
    The quantum part (period finding) is done classically here.
    """
    if verbose:
        print(f"Shor's Algorithm Simulation: Factoring N = {N}")
        print("=" * 50)
    
    for attempt in range(20):
        a = random.randint(2, N - 1)
        
        g = gcd(a, N)
        if g > 1:
            if verbose:
                print(f"Attempt {attempt+1}: a={a}, gcd({a},{N})={g} — Lucky!")
            return g, N // g
        
        # Find period (this is what the quantum computer does)
        r = 1
        while pow(a, r, N) != 1:
            r += 1
            if r > N:
                break
        
        if verbose:
            print(f"Attempt {attempt+1}: a={a}, r={r}", end="")
        
        if r % 2 != 0:
            if verbose:
                print(" — r is odd, retry")
            continue
        
        x = pow(a, r // 2, N)
        if x == N - 1:
            if verbose:
                print(f" — a^(r/2) ≡ -1 (mod N), retry")
            continue
        
        f1 = gcd(x - 1, N)
        f2 = gcd(x + 1, N)
        
        if f1 not in [1, N] and f2 not in [1, N]:
            if verbose:
                print(f" — Success! Factors: {f1} × {f2}")
            return min(f1, f2), max(f1, f2)
        else:
            if verbose:
                print(f" — trivial factors ({f1}, {f2}), retry")
    
    return None, None

# Factor 21
print("Factoring N = 21:")
p, q = classical_shor_simulation(21)
print(f"\n21 = {p} × {q}")

print("\n" + "=" * 50)
print("\nFactoring N = 35:")
p, q = classical_shor_simulation(35)
print(f"\n35 = {p} × {q}")

print("\n" + "=" * 50)
print("\nFactoring N = 77:")
p, q = classical_shor_simulation(77)
print(f"\n77 = {p} × {q}")

## 9. Quantum Period Finding — Detailed Analysis

Let us understand exactly what the quantum Fourier transform does to the periodic state.

### The State Before QFT

After the controlled modular exponentiation and measurement of the work register (conceptually), the counting register is in a superposition of states that are multiples of $Q/r$:

$$|\psi\rangle = \frac{1}{\sqrt{A}} \sum_{j=0}^{A-1} |s_0 + jr\rangle$$

where $A \approx Q/r$ is the number of terms and $s_0$ is some offset.

### After QFT$^\dagger$

The inverse QFT transforms this periodic state into peaks at:

$$m \approx k \cdot \frac{Q}{r}, \quad k = 0, 1, \ldots, r-1$$

The probability of measuring each peak is approximately $1/r$, so we get a uniformly random multiple of $Q/r$.

In [ ]:
# Visualize the expected measurement distribution for different r values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

Q = 256  # 2^8
for idx, (a, r) in enumerate([(7, 4), (2, 4), (4, 2)]):
    # Expected peaks at k * Q/r
    peaks = [int(round(k * Q / r)) % Q for k in range(r)]
    
    x = range(Q)
    y = np.zeros(Q)
    for p in peaks:
        y[p] = 1.0 / r  # approximate probability
    
    axes[idx].bar(x, y, color='steelblue', alpha=0.8, width=1.0)
    axes[idx].set_xlabel('Measured value m', fontsize=11)
    axes[idx].set_ylabel('Probability', fontsize=11)
    axes[idx].set_title(f'a={a}, r={r}: peaks at m={peaks}', fontsize=12)
    axes[idx].set_xlim(-5, Q + 5)

plt.suptitle(f'Expected QFT Output (Q={Q}, N=15)', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Connection to QFT and Phase Estimation

Shor's algorithm is actually an instance of the **Quantum Phase Estimation** (QPE) algorithm!

The unitary $U_a|y\rangle = |ay \mod N\rangle$ has eigenstates:

$$|u_s\rangle = \frac{1}{\sqrt{r}} \sum_{k=0}^{r-1} e^{-2\pi i sk/r} |a^k \mod N\rangle, \quad s = 0, 1, \ldots, r-1$$

with eigenvalues:

$$U_a |u_s\rangle = e^{2\pi i s/r} |u_s\rangle$$

QPE estimates the phase $s/r$, from which we extract $r$ using continued fractions.

The beautiful fact: $|1\rangle = \frac{1}{\sqrt{r}} \sum_{s=0}^{r-1} |u_s\rangle$, so initializing the work register to $|1\rangle$ automatically puts us in an equal superposition of all eigenstates — no need to prepare individual eigenstates!

In [ ]:
# Verify the eigenstate decomposition for a=7, N=15, r=4
a = 7
N = 15
r = 4

print(f"Eigenstate decomposition for a={a}, N={N}, r={r}")
print("=" * 50)

# The cycle: a^k mod N for k = 0, 1, ..., r-1
cycle = [pow(a, k, N) for k in range(r)]
print(f"Cycle: {cycle}")
print()

# Eigenvalues
for s in range(r):
    eigenvalue = np.exp(2j * np.pi * s / r)
    phase = s / r
    print(f"Eigenstate |u_{s}⟩: eigenvalue = e^(2πi·{s}/{r}), phase = {phase:.4f}")
    
    # Verify: U_a |u_s⟩ = e^(2πis/r) |u_s⟩
    # The eigenstate coefficients
    coeffs = [np.exp(-2j * np.pi * s * k / r) / np.sqrt(r) for k in range(r)]
    print(f"  Coefficients: {[f'{c:.3f}' for c in coeffs]}")
    print(f"  States:       {cycle}")

print(f"\n|1⟩ = (1/√{r}) Σ_s |u_s⟩ — verified: initializing to |1⟩ is correct!")

## 11. Resource Estimates for Practical Factoring

How many qubits and gates are needed to factor real RSA numbers?

In [ ]:
# Resource estimates
print("Shor's Algorithm — Resource Estimates")
print("=" * 70)
print(f"{'RSA Key':<15} {'Bits (n)':<10} {'Qubits':<12} {'T-gates':<18} {'Est. Runtime'}")
print("-" * 70)

# Approximate resource estimates based on literature
rsa_sizes = [
    ('RSA-512', 512),
    ('RSA-1024', 1024),
    ('RSA-2048', 2048),
    ('RSA-4096', 4096),
]

for name, n_bits in rsa_sizes:
    # Approximate qubit count: ~2n + O(n) ancillas
    qubits = 3 * n_bits + 1  # approximate
    # T-gate count: O(n^3) with optimizations
    t_gates = n_bits ** 3  # rough order
    # Runtime estimate (very rough, assuming 1μs T-gate)
    runtime_s = t_gates * 1e-6
    if runtime_s < 60:
        runtime_str = f"~{runtime_s:.1f} s"
    elif runtime_s < 3600:
        runtime_str = f"~{runtime_s/60:.1f} min"
    else:
        runtime_str = f"~{runtime_s/3600:.1f} hr"
    
    print(f"{name:<15} {n_bits:<10} {qubits:<12} {t_gates:<18,.0f} {runtime_str}")

print("\nNote: These are rough estimates. Actual implementations vary widely.")
print("Current record: Factored 21 on a real quantum computer (IBM, 2001).")
print("State of the art (2025): ~1000 logical qubits needed for RSA-2048.")

## 12. Summary of Shor's Algorithm

### The Complete Algorithm

1. **Input:** $N$ (number to factor)
2. **Check:** Is $N$ even? Is $N = a^b$ for some $a, b$? (Trivial cases)
3. **Choose** random $a$ with $1 < a < N$
4. **Compute** $\gcd(a, N)$. If $> 1$, we found a factor.
5. **Quantum Period Finding:** Find the order $r$ of $a \mod N$
   - Prepare $|0\rangle^{\otimes t}|1\rangle$
   - Apply $H^{\otimes t}$ to counting register
   - Apply controlled-$U_a^{2^j}$ for $j = 0, 1, \ldots, t-1$
   - Apply QFT$^\dagger$ to counting register
   - Measure $\to m$
   - Use continued fractions: $m/2^t \approx s/r \to r$
6. **Classical post-processing:** If $r$ is even and $a^{r/2} \not\equiv -1$:
   - $\gcd(a^{r/2} \pm 1, N)$ gives factors
7. **Repeat** if unsuccessful

### Complexity

- **Quantum:** $O((\log N)^2 \log\log N)$ quantum gates, $O(\log N)$ qubits
- **Classical post-processing:** $O((\log N)^3)$ operations
- **Total:** $O((\log N)^3)$ — polynomial in the number of digits!

### Impact

- Breaks RSA, DSA, ECC, and all factoring/discrete-log-based cryptography
- Motivates the transition to **post-quantum cryptography** (lattice-based, hash-based, etc.)
- NIST finalized PQC standards in 2024: CRYSTALS-Kyber, CRYSTALS-Dilithium, SPHINCS+

## 13. Exercises

1. **Verify classically** that for $N = 21$, $a = 2$: the period is $r = 6$, and $\gcd(2^3 - 1, 21) = 7$, $\gcd(2^3 + 1, 21) = 3$.

2. **Implement** the continued fraction algorithm from scratch (without `Fraction.limit_denominator`) and verify it gives the correct period for $m/Q$ values.

3. **Failure probability:** For $N = 15$, enumerate all possible $(a, r)$ pairs. What fraction lead to successful factoring?

4. **Circuit optimization:** For the $a = 7, N = 15$ case, count the number of CNOT gates in the compiled circuit. How does this compare to the theoretical $O(n^3)$ bound?

5. **Think about:** Why does Shor's algorithm not violate the $\Omega(\sqrt{N})$ lower bound for unstructured search (Grover's optimality)?

**Next:** [Notebook 09 — Quantum Fourier Transform](09-quantum-fourier-transform.ipynb)